In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 1 │ Configuration
# ──────────────────────────────────────────────────────────────
SILVER_PATH = "/Volumes/workspace/fraud_platform/data/silver/transactions"
GOLD_PATH   = "/Volumes/workspace/fraud_platform/data/gold"

print("✅ Gold config loaded")

✅ Gold config loaded


In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 2 │ Gold Table 1 — Card Risk Summary
# One row per card: their overall risk profile
# ──────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

silver = spark.read.format("delta").load(SILVER_PATH)

card_risk = (
    silver
    .groupBy("card_id", "customer_id")
    .agg(
        F.count("transaction_id")                    .alias("total_txns"),
        F.round(F.sum("amount_local"), 2)            .alias("total_spent_eur"),
        F.round(F.avg("amount_local"), 2)            .alias("avg_txn_amount"),
        F.max("amount_local")                        .alias("max_txn_amount"),
        F.round(F.avg("risk_score"), 1)              .alias("avg_risk_score"),
        F.max("risk_score")                          .alias("max_risk_score"),
        F.sum(F.col("is_high_amount").cast("int"))   .alias("high_amount_count"),
        F.sum(F.col("is_foreign_country").cast("int")).alias("foreign_txn_count"),
        F.sum(F.col("is_suspicious_mcc").cast("int")).alias("suspicious_mcc_count"),
        F.sum(F.col("is_rapid_succession").cast("int")).alias("rapid_succession_count"),
        F.max("event_ts")                            .alias("last_seen_at"),
    )
    .withColumn("card_risk_label",
        F.when(F.col("max_risk_score") >= 60, "HIGH")
         .when(F.col("max_risk_score") >= 30, "MEDIUM")
         .otherwise("LOW")
    )
    .orderBy(F.col("max_risk_score").desc())
)

card_risk.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/card_risk_summary")
print(f"✅ Card risk summary written — {card_risk.count()} cards")
card_risk.filter(F.col("card_risk_label") == "HIGH").show(truncate=False)

✅ Card risk summary written — 50 cards
+---------+-----------+----------+---------------+--------------+--------------+--------------+--------------+-----------------+-----------------+--------------------+----------------------+-----------------------+---------------+
|card_id  |customer_id|total_txns|total_spent_eur|avg_txn_amount|max_txn_amount|avg_risk_score|max_risk_score|high_amount_count|foreign_txn_count|suspicious_mcc_count|rapid_succession_count|last_seen_at           |card_risk_label|
+---------+-----------+----------+---------------+--------------+--------------+--------------+--------------+-----------------+-----------------+--------------------+----------------------+-----------------------+---------------+
|CARD_0030|CUST_0030  |39        |9327.33        |239.16        |2513.34       |16.7          |80            |1                |6                |3                   |37                    |2026-08-20 19:46:12.938|HIGH           |
|CARD_0004|CUST_0004  |33        |146

In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 3 │ Gold Table 2 — Fraud Alerts
# One row per HIGH risk transaction — what an analyst acts on
# ──────────────────────────────────────────────────────────────
alerts = (
    silver
    .filter(F.col("risk_label") == "HIGH")
    .select(
        "transaction_id", "card_id", "customer_id",
        "event_ts", "merchant_name", "merchant_category_code",
        "amount_local", "currency_code", "country_code",
        "terminal_type", "risk_score", "risk_label",
        "is_high_amount", "is_foreign_country",
        "is_suspicious_mcc", "is_rapid_succession",
        "txn_count_5min", "processed_at"
    )
    .withColumn("alert_reason",
        F.concat_ws(" + ",
            F.when(F.col("is_high_amount"),       F.lit("HIGH_AMOUNT")),
            F.when(F.col("is_foreign_country"),   F.lit("FOREIGN_COUNTRY")),
            F.when(F.col("is_suspicious_mcc"),    F.lit("SUSPICIOUS_MCC")),
            F.when(F.col("is_rapid_succession"),  F.lit("RAPID_SUCCESSION")),
        )
    )
    .orderBy(F.col("risk_score").desc(), F.col("event_ts").desc())
)

alerts.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/fraud_alerts")
print(f"🚨 {alerts.count()} fraud alerts written")
alerts.show(truncate=False)

🚨 7 fraud alerts written
+------------------------------------+---------+-----------+-----------------------+----------------+----------------------+------------+-------------+------------+-------------+----------+----------+--------------+------------------+-----------------+-------------------+--------------+--------------------------+---------------------------------------------------+
|transaction_id                      |card_id  |customer_id|event_ts               |merchant_name   |merchant_category_code|amount_local|currency_code|country_code|terminal_type|risk_score|risk_label|is_high_amount|is_foreign_country|is_suspicious_mcc|is_rapid_succession|txn_count_5min|processed_at              |alert_reason                                       |
+------------------------------------+---------+-----------+-----------------------+----------------+----------------------+------------+-------------+------------+-------------+----------+----------+--------------+------------------+-------